<a href="https://colab.research.google.com/github/ONESFA/PINNs-and-FDM-FEM/blob/main/%EC%86%90%EC%8B%A4%ED%95%A8%EC%88%98_%EA%B5%AC%ED%98%84_%EC%99%84.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

필요한 모듈 설치


In [ ]:
import torch

1. Loss 다 합친거 함수


In [ ]:
def loss_pde(model, x_f, y_f, t_f, c=1.0): #c는 상수이므로 1로 가정

    residual = pde_residual(
        model,
        x_f,
        y_f,
        t_f,
        c
    )

    L_pde = torch.mean(residual ** 2)

    return L_pde

Initial Displacement Loss

In [ ]:
def loss_ic(model, x_ic, y_ic, u_ic):

    t_ic = torch.zeros_like(x_ic)

    u_pred = model(
        x_ic,
        y_ic,
        t_ic
    )

    L_ic = torch.mean(
        (u_pred - u_ic) ** 2
    )

    return L_ic

Initial Velocity Loss

In [ ]:
def loss_ic_velocity(model, x_ic, y_ic, v_ic):

    t_ic = torch.zeros_like(x_ic)
    t_ic.requires_grad_(True)

    u_pred = model(
        x_ic,
        y_ic,
        t_ic
    )

    u_t = torch.autograd.grad(
        u_pred,
        t_ic,
        grad_outputs=torch.ones_like(u_pred),
        create_graph=True
    )[0]

    L_ic_velocity = torch.mean(
        (u_t - v_ic) ** 2
    )

    return L_ic_velocity

Boundary Condition Loss

In [ ]:
def loss_bc(model, x_bc, y_bc, t_bc):

    u_pred = model(
        x_bc,
        y_bc,
        t_bc
    )

    L_bc = torch.mean(
        u_pred ** 2
    )

    return L_bc

Total

In [ ]:
def total_loss(
    model,
    x_f, y_f, t_f,
    x_bc, y_bc, t_bc,
    x_ic, y_ic,
    u_ic,
    v_ic,
    c=1.0
):

    L_pde = loss_pde(
        model,
        x_f, y_f, t_f,
        c
    )

    L_bc = loss_bc(
        model,
        x_bc, y_bc, t_bc
    )

    L_ic = loss_ic(
        model,
        x_ic, y_ic,
        u_ic
    )

    L_ic_velocity = loss_ic_velocity(
        model,
        x_ic, y_ic,
        v_ic
    )

    L_loss = (
        L_pde
        + L_bc
        + L_ic
        + L_ic_velocity
    )

    return (
        L_loss,
        L_pde,
        L_bc,
        L_ic,
        L_ic_velocity
    )

In [ ]:
N_f = 100  #점 100개씩 검사
N_bc = 100
N_ic = 100

# PDE 내부 점
x_f = torch.rand(N_f, 1)  #랜덤으로 점을 뽑아서 계산한다. 주어진 범위가 0부터 1이므로 rand 함수로 0부터 1까지의 랜덤값을 불러온다
y_f = torch.rand(N_f, 1)
t_f = torch.rand(N_f, 1)

# Boundary 점
x_bc = torch.rand(N_bc, 1)
y_bc = torch.rand(N_bc, 1)
t_bc = torch.rand(N_bc, 1)

# Initial Condition 점
x_ic = torch.rand(N_ic, 1)
y_ic = torch.rand(N_ic, 1)

# 초기 변위
u_ic = torch.sin(torch.pi * x_ic) * torch.sin(torch.pi * y_ic)

# 초기 속도
v_ic = torch.zeros_like(u_ic)

손실 함수들의 실행 코드


In [ ]:
L_loss, L_pde, L_bc, L_ic, L_ic_velocity = total_loss(
    model,

    x_f, y_f, t_f,

    x_bc, y_bc, t_bc,

    x_ic, y_ic,
    u_ic,
    v_ic
)

각각 함수들의 출력값 확인

In [ ]:
print("Total Loss     =", L_loss.item())
print("PDE Loss       =", L_pde.item())
print("BC Loss        =", L_bc.item())
print("IC Loss        =", L_ic.item())
print("IC Velocity    =", L_ic_velocity.item())